In [10]:
%%writefile requirements.txt
numpy
matplotlib
gymnasium
torch
tensorboard
sb3-contrib
tqdm

Overwriting requirements.txt


In [11]:
!pip3 install -r requirements.txt

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


In [12]:
%%writefile src/direct_guidance_fighter_env.py
from __future__ import annotations

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class DirectGuidanceFighterEnv(gym.Env):
    metadata = {"render_modes": []}

    MODE_RECOVER = 0
    MODE_EVADE = 1
    MODE_ATTACK = 2
    MODE_PURSUIT = 3
    NUM_MODES = 4

    def __init__(self):
        super().__init__()

        self.dt = 0.05
        self.max_steps = 1500
        self.g = 9.81

        self.min_combat_altitude = 25.0
        self.safe_altitude = 42.0

        self.base_obs_dim = 21
        self.obs_dim = self.base_obs_dim + self.NUM_MODES
        self.act_dim = 4

        self.damage_scale = 8.0

        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0, -1.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.obs_dim,),
            dtype=np.float32,
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.pos = np.array([
            [0.0, 0.0, 50.0],
            [80.0, 25.0, 50.0],
        ], dtype=np.float32)

        self.vel = np.array([
            [30.0, 0.0, 0.0],
            [-22.0, 0.0, 0.0],
        ], dtype=np.float32)

        self.forward_vec = np.array([
            [1.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0],
        ], dtype=np.float32)

        self.pos += self.np_random.uniform(-5.0, 5.0, size=self.pos.shape).astype(np.float32)
        self.pos[:, 2] = np.clip(self.pos[:, 2], 42.0, 60.0)

        self.enemy_turn_dir = float(self.np_random.choice([-1.0, 1.0]))

        self.hp = np.array([100.0, 100.0], dtype=np.float32)
        self.step_count = 0
        self.current_mode = self.MODE_PURSUIT

        return self._get_obs(0), {}

    def _normalize(self, v):
        n = np.linalg.norm(v)
        if n < 1e-6:
            return np.array([1.0, 0.0, 0.0], dtype=np.float32)
        return (v / n).astype(np.float32)

    def _forward(self, i):
        return self.forward_vec[i]

    def _right(self, i):
        f = self._forward(i)
        world_up = np.array([0.0, 0.0, 1.0], dtype=np.float32)
        r = np.cross(f, world_up)

        if np.linalg.norm(r) < 1e-6:
            r = np.array([0.0, 1.0, 0.0], dtype=np.float32)

        return self._normalize(r)

    def _up(self, i):
        r = self._right(i)
        f = self._forward(i)
        return self._normalize(np.cross(r, f))

    def _speed(self, i):
        return float(np.linalg.norm(self.vel[i]))

    def _energy(self, i):
        height = max(float(self.pos[i, 2]), 0.0)
        speed = self._speed(i)
        return self.g * height + 0.5 * speed ** 2

    def _los_angle_deg(self, shooter, target):
        rel = self.pos[target] - self.pos[shooter]
        dist = float(np.linalg.norm(rel))
        to_target = rel / (dist + 1e-6)

        forward = self._forward(shooter)
        cos_theta = np.clip(np.dot(forward, to_target), -1.0, 1.0)
        theta = float(np.degrees(np.arccos(cos_theta)))

        return theta, dist

    def _compute_damage_rate(self, dist, theta_deg):
        if 5.0 <= dist <= 50.0 and theta_deg < 10.0:
            los_factor = 1.0 - theta_deg / 10.0
            range_factor = 1.0 - (dist - 5.0) / 45.0
            return 1.5 * max(los_factor, 0.0) * max(range_factor, 0.0)

        if 5.0 <= dist <= 80.0 and theta_deg < 25.0:
            los_factor = 1.0 - theta_deg / 25.0
            range_factor = 1.0 - (dist - 5.0) / 75.0
            return 0.6 * max(los_factor, 0.0) * max(range_factor, 0.0)

        if 5.0 <= dist <= 120.0 and theta_deg < 45.0:
            los_factor = 1.0 - theta_deg / 45.0
            range_factor = 1.0 - (dist - 5.0) / 115.0
            return 0.25 * max(los_factor, 0.0) * max(range_factor, 0.0)

        return 0.0

    def _is_enemy_behind(self, i, enemy):
        rel = self.pos[enemy] - self.pos[i]
        dist = np.linalg.norm(rel)
        to_enemy = rel / (dist + 1e-6)

        return float(np.dot(self._forward(i), to_enemy)) < -0.15

    def _enemy_on_my_six_score(self, i, enemy):
        rel = self.pos[enemy] - self.pos[i]
        dist = np.linalg.norm(rel)

        to_enemy = rel / (dist + 1e-6)
        to_me = -to_enemy

        enemy_behind_me = -np.dot(self._forward(i), to_enemy)
        enemy_aiming_me = np.dot(self._forward(enemy), to_me)

        return float(max(0.0, enemy_behind_me) * max(0.0, enemy_aiming_me))

    def _pseudo_rpy(self, i):
        f = self._forward(i)
        yaw = np.arctan2(f[1], f[0])
        pitch = np.arctan2(f[2], np.linalg.norm(f[:2]) + 1e-6)
        roll = 0.0
        return np.array([roll, pitch, yaw], dtype=np.float32)

    def _select_mode(self):
        height = float(self.pos[0, 2])
        speed = self._speed(0)

        dist = float(np.linalg.norm(self.pos[1] - self.pos[0]))
        agent_los_deg, _ = self._los_angle_deg(0, 1)

        enemy_behind = self._is_enemy_behind(0, 1)
        danger_score = self._enemy_on_my_six_score(0, 1)

        if height < self.safe_altitude + 5.0 or speed < 16.0:
            return self.MODE_RECOVER

        if enemy_behind and danger_score > 0.65:
            return self.MODE_EVADE

        if 5.0 < dist < 120.0 and agent_los_deg < 70.0:
            return self.MODE_ATTACK

        return self.MODE_PURSUIT

    def _mode_one_hot(self, mode):
        x = np.zeros(self.NUM_MODES, dtype=np.float32)
        x[mode] = 1.0
        return x

    def _get_base_obs(self, i):
        j = 1 - i

        rel_pos = self.pos[j] - self.pos[i]
        rel_vel = self.vel[j] - self.vel[i]

        my_speed = self._speed(i)
        enemy_speed = self._speed(j)
        my_energy = self._energy(i)
        enemy_energy = self._energy(j)

        obs = np.concatenate([
            self.pos[i] / 100.0,
            self.vel[i] / 100.0,
            self._pseudo_rpy(i),
            rel_pos / 100.0,
            rel_vel / 100.0,
            np.array([
                my_speed / 100.0,
                my_energy / 5000.0,
                enemy_speed / 100.0,
                enemy_energy / 5000.0,
                self.hp[i] / 100.0,
                self.hp[j] / 100.0,
            ], dtype=np.float32),
        ])

        return obs.astype(np.float32)

    def _get_obs(self, i):
        if i == 0:
            self.current_mode = self._select_mode()
            mode_one_hot = self._mode_one_hot(self.current_mode)
        else:
            mode_one_hot = self._mode_one_hot(self.MODE_PURSUIT)

        return np.concatenate([self._get_base_obs(i), mode_one_hot]).astype(np.float32)

    def _apply_guidance_action(self, i, action):
        dir_cmd = np.asarray(action[:3], dtype=np.float32)
        throttle = float(action[3])

        height = float(self.pos[i, 2])
        vertical_speed = float(self.vel[i, 2])

        desired_dir = self._normalize(dir_cmd)

        if height < self.safe_altitude:
            desired_dir[2] = max(desired_dir[2], 0.65)
            throttle = 1.0

        if height < self.safe_altitude + 8.0:
            desired_dir[2] = max(desired_dir[2], 0.35)
            throttle = max(throttle, 0.9)

        if vertical_speed < -3.0:
            desired_dir[2] = max(desired_dir[2], 0.45)
            throttle = max(throttle, 0.95)

        desired_dir[2] = np.clip(desired_dir[2], -0.15, 0.85)
        desired_dir = self._normalize(desired_dir)

        turn_alpha = 0.12
        new_forward = self._normalize(
            (1.0 - turn_alpha) * self.forward_vec[i] + turn_alpha * desired_dir
        )
        self.forward_vec[i] = new_forward

        throttle = 0.35 + 0.65 * throttle
        throttle = np.clip(throttle, 0.35, 1.0)

        speed = self._speed(i)

        thrust_accel = self.forward_vec[i] * throttle * 42.0
        gravity = np.array([0.0, 0.0, -self.g], dtype=np.float32)
        drag = -0.008 * self.vel[i] * speed

        accel = thrust_accel + gravity + drag

        self.vel[i] += accel * self.dt
        self.pos[i] += self.vel[i] * self.dt

    def _enemy_constant_turn_action(self):
        f = self._forward(1)
        yaw = np.arctan2(f[1], f[0])
        yaw += self.enemy_turn_dir * 0.35 * self.dt

        height = float(self.pos[1, 2])
        desired_z = 0.0
        throttle = 0.68

        if height < self.safe_altitude:
            desired_z = 0.5
            throttle = 1.0
        elif height > 70.0:
            desired_z = -0.25
            throttle = 0.55

        desired_dir = np.array([
            np.cos(yaw),
            np.sin(yaw),
            desired_z,
        ], dtype=np.float32)

        return np.array([
            desired_dir[0],
            desired_dir[1],
            desired_dir[2],
            throttle,
        ], dtype=np.float32)

    def _mode_reward(
        self,
        mode,
        dist,
        progress,
        heading_align,
        closing_speed,
        lateral_error,
        vertical_error,
        dir_cmd,
        agent_los_deg,
        enemy_damage_rate,
        agent_damage_rate,
        height,
        speed,
        danger_score,
        enemy_behind_agent,
    ):
        reward = 0.0
        los_reward = max(0.0, 1.0 - agent_los_deg / 60.0)

        desired_dir = self._normalize(dir_cmd)
        to_enemy = self._normalize(self.pos[1] - self.pos[0])

        reward += 4.0 * float(np.dot(desired_dir, to_enemy))

        if mode == self.MODE_RECOVER:
            reward -= 1.2 * abs(height - self.safe_altitude)

            if height >= self.safe_altitude:
                reward += 20.0

            if height >= self.safe_altitude + 8.0:
                reward += 10.0

            if 24.0 < speed < 90.0:
                reward += 3.0

            reward -= 80.0 * enemy_damage_rate

        elif mode == self.MODE_EVADE:
            reward += 4.0 * danger_score

            if enemy_behind_agent and progress < 0.0:
                reward += 5.0 * (-progress)

            reward -= 100.0 * enemy_damage_rate

            if height >= self.safe_altitude:
                reward += 10.0

            if 30.0 < speed < 100.0:
                reward += 1.0

        elif mode == self.MODE_ATTACK:
            reward += 220.0 * agent_damage_rate
            reward -= 120.0 * enemy_damage_rate

            reward += 12.0 * heading_align
            reward += 10.0 * los_reward
            reward += 2.0 * progress

            # Good firing geometry reward.
            if agent_los_deg < 45.0 and 5.0 < dist < 120.0:
                reward += 5.0

            if agent_los_deg < 25.0 and 5.0 < dist < 80.0:
                reward += 10.0

            if agent_los_deg < 10.0 and 5.0 < dist < 50.0:
                reward += 20.0

            if 8.0 < dist < 65.0:
                reward += 5.0

            if dist < 5.0:
                reward -= 10.0

        elif mode == self.MODE_PURSUIT:
            reward += 10.0 * heading_align
            reward += 6.0 * los_reward
            reward += 6.0 * progress
            reward += 0.04 * closing_speed
            reward -= 0.020 * dist

            if 15.0 < dist < 100.0:
                reward += 2.0

            reward += 80.0 * agent_damage_rate
            reward -= 80.0 * enemy_damage_rate

        return reward

    def step(self, action):
        mode = self._select_mode()
        self.current_mode = mode

        agent_action = np.asarray(action, dtype=np.float32)
        enemy_action = self._enemy_constant_turn_action()

        prev_dist = float(np.linalg.norm(self.pos[1] - self.pos[0]))

        self._apply_guidance_action(0, agent_action)
        self._apply_guidance_action(1, enemy_action)

        self.step_count += 1

        dist = float(np.linalg.norm(self.pos[1] - self.pos[0]))
        progress = prev_dist - dist

        agent_los_deg, agent_dist = self._los_angle_deg(0, 1)
        enemy_los_deg, enemy_dist = self._los_angle_deg(1, 0)

        agent_damage_rate = self._compute_damage_rate(agent_dist, agent_los_deg)
        enemy_damage_rate = self._compute_damage_rate(enemy_dist, enemy_los_deg)

        agent_damage_to_hp = agent_damage_rate * self.damage_scale
        enemy_damage_to_hp = enemy_damage_rate * self.damage_scale

        self.hp[1] -= agent_damage_to_hp
        self.hp[0] -= enemy_damage_to_hp
        self.hp = np.clip(self.hp, 0.0, 100.0)

        speed = self._speed(0)
        height = float(self.pos[0, 2])
        enemy_height = float(self.pos[1, 2])

        rel = self.pos[1] - self.pos[0]
        to_enemy = rel / (np.linalg.norm(rel) + 1e-6)

        forward = self._forward(0)
        right = self._right(0)
        up = self._up(0)

        heading_align = float(np.dot(forward, to_enemy))
        closing_speed = float(np.dot(self.vel[0], to_enemy))
        lateral_error = float(np.dot(right, to_enemy))
        vertical_error = float(np.dot(up, to_enemy))

        enemy_behind_agent = self._is_enemy_behind(0, 1)
        danger_score = self._enemy_on_my_six_score(0, 1)

        reward = self._mode_reward(
            mode=mode,
            dist=dist,
            progress=progress,
            heading_align=heading_align,
            closing_speed=closing_speed,
            lateral_error=lateral_error,
            vertical_error=vertical_error,
            dir_cmd=agent_action[:3],
            agent_los_deg=agent_los_deg,
            enemy_damage_rate=enemy_damage_rate,
            agent_damage_rate=agent_damage_rate,
            height=height,
            speed=speed,
            danger_score=danger_score,
            enemy_behind_agent=enemy_behind_agent,
        )

        vertical_speed = float(self.vel[0, 2])

        altitude_error = max(0.0, self.safe_altitude - height)
        reward -= 8.0 * altitude_error

        if height >= self.safe_altitude:
            reward += 5.0

        if height >= self.safe_altitude + 10.0:
            reward += 2.0

        if height < self.min_combat_altitude + 5.0:
            reward -= 100.0

        if vertical_speed < -3.0:
            reward -= 2.0 * abs(vertical_speed)

        if speed < 10.0:
            reward -= 8.0

        if speed > 160.0:
            reward -= 12.0

        reward -= 0.02

        terminated = False
        truncated = False

        if self.hp[1] <= 0.0:
            reward += 300.0
            terminated = True

        if self.hp[0] <= 0.0:
            reward -= 300.0
            terminated = True

        if height < self.min_combat_altitude:
            reward -= 300.0
            terminated = True

        if enemy_height < self.min_combat_altitude:
            reward += 300.0
            terminated = True

        if self.step_count >= self.max_steps:
            truncated = True

        info = {
            "mode": int(mode),
            "dist": dist,
            "agent_hp": float(self.hp[0]),
            "enemy_hp": float(self.hp[1]),
            "agent_damage_rate": float(agent_damage_rate),
            "enemy_damage_rate": float(enemy_damage_rate),
            "agent_damage_to_hp": float(agent_damage_to_hp),
            "enemy_damage_to_hp": float(enemy_damage_to_hp),
            "agent_damage": float(agent_damage_rate),
            "enemy_damage": float(enemy_damage_rate),
            "agent_los_deg": float(agent_los_deg),
            "enemy_los_deg": float(enemy_los_deg),
            "agent_energy": float(self._energy(0)),
            "enemy_energy": float(self._energy(1)),
            "agent_speed": float(speed),
            "enemy_speed": float(self._speed(1)),
            "agent_pos": self.pos[0].copy(),
            "enemy_pos": self.pos[1].copy(),
            "agent_action": agent_action.copy(),
            "enemy_action": enemy_action.copy(),
            "danger_score": float(danger_score),
            "agent_vertical_speed": vertical_speed,
            "agent_altitude": float(height),
            "enemy_altitude": float(enemy_height),
            "min_combat_altitude": float(self.min_combat_altitude),
            "safe_altitude": float(self.safe_altitude),
        }

        return self._get_obs(0), float(reward), terminated, truncated, info

Overwriting src/direct_guidance_fighter_env.py


In [ ]:
import os
import torch

from tqdm.auto import tqdm

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecMonitor

from src.direct_guidance_fighter_env import DirectGuidanceFighterEnv


class TQDMProgressBarCallback(BaseCallback):
    def __init__(self, total_timesteps: int):
        super().__init__()
        self.total_timesteps = total_timesteps
        self.pbar = None

    def _on_training_start(self):
        self.pbar = tqdm(
            total=self.total_timesteps,
            desc="Training",
            dynamic_ncols=True,
        )

    def _on_step(self):
        self.pbar.n = min(self.num_timesteps, self.total_timesteps)

        if len(self.model.ep_info_buffer) > 0:
            ep_rew_mean = sum(e["r"] for e in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)
            ep_len_mean = sum(e["l"] for e in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)

            self.pbar.set_postfix({
                "reward": f"{ep_rew_mean:.2f}",
                "len": f"{ep_len_mean:.0f}",
            })

        self.pbar.refresh()
        return True

    def _on_training_end(self):
        if self.pbar is not None:
            self.pbar.n = self.total_timesteps
            self.pbar.refresh()
            self.pbar.close()


def make_env(seed: int):
    def _init():
        env = DirectGuidanceFighterEnv()
        env.reset(seed=seed)
        return Monitor(env)
    return _init


os.makedirs("models", exist_ok=True)
os.makedirs("logs/recurrent_ppo_direct_guidance", exist_ok=True)

N_ENVS = 8
TOTAL_TIMESTEPS = 500_000
USE_SUBPROC = True

if USE_SUBPROC:
    env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
else:
    env = DummyVecEnv([make_env(i) for i in range(N_ENVS)])

env = VecMonitor(env)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = RecurrentPPO(
    policy="MlpLstmPolicy",
    env=env,
    device=device,
    verbose=0,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=512,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,
    vf_coef=0.5,
    max_grad_norm=0.5,
    tensorboard_log="logs/recurrent_ppo_direct_guidance",
)

callback = TQDMProgressBarCallback(
    total_timesteps=TOTAL_TIMESTEPS,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
)

model.save("models/recurrent_ppo_direct_guidance")

env.close()

print("saved: models/recurrent_ppo_direct_guidance.zip")

/Users/br4c3/Projects/aitopgun-rl/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/br4c3/Projects/aitopgun-rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


device: cpu


Training:  18%|█▊        | 89032/500000 [02:42<12:28, 548.96it/s, reward=-42821.43, len=1500]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sb3_contrib import RecurrentPPO
from src.direct_guidance_fighter_env import DirectGuidanceFighterEnv


MODE_NAMES = {
    0: "RECOVER",
    1: "EVADE",
    2: "ATTACK",
    3: "PURSUIT",
}

MAX_COMBAT_STEPS = 100_000
DETERMINISTIC = True

env = DirectGuidanceFighterEnv()
env.max_steps = 10**12

model = RecurrentPPO.load("models/recurrent_ppo_direct_guidance")

obs, _ = env.reset()

lstm_states = None
episode_starts = np.ones((1,), dtype=bool)

agent_traj = []
enemy_traj = []
modes = []
actions = []

agent_hp = []
enemy_hp = []

agent_damage_rate = []
enemy_damage_rate = []
agent_damage_to_hp = []
enemy_damage_to_hp = []

agent_los = []
enemy_los = []
dists = []
agent_energy = []
enemy_energy = []
rewards = []

firing_events = []

winner = None

for step in range(MAX_COMBAT_STEPS):
    action, lstm_states = model.predict(
        obs,
        state=lstm_states,
        episode_start=episode_starts,
        deterministic=DETERMINISTIC,
    )

    obs, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated
    episode_starts = np.array([done], dtype=bool)

    agent_traj.append(info["agent_pos"])
    enemy_traj.append(info["enemy_pos"])
    modes.append(info["mode"])
    actions.append(action.copy())

    agent_hp.append(info["agent_hp"])
    enemy_hp.append(info["enemy_hp"])

    agent_damage_rate.append(info["agent_damage_rate"])
    enemy_damage_rate.append(info["enemy_damage_rate"])
    agent_damage_to_hp.append(info["agent_damage_to_hp"])
    enemy_damage_to_hp.append(info["enemy_damage_to_hp"])

    agent_los.append(info["agent_los_deg"])
    enemy_los.append(info["enemy_los_deg"])

    dists.append(info["dist"])
    agent_energy.append(info["agent_energy"])
    enemy_energy.append(info["enemy_energy"])
    rewards.append(reward)

    if info["agent_damage_to_hp"] > 0.0:
        firing_events.append({
            "agent_pos": info["agent_pos"].copy(),
            "enemy_pos": info["enemy_pos"].copy(),
            "damage": info["agent_damage_to_hp"],
        })

    if info["enemy_hp"] <= 0.0:
        winner = "Agent wins"
        print(f"{winner} at step {step}")
        break

    if info["agent_hp"] <= 0.0:
        winner = "Enemy wins"
        print(f"{winner} at step {step}")
        break

    if info["agent_altitude"] < info["min_combat_altitude"]:
        winner = "Agent crashed / below combat altitude"
        print(f"{winner} at step {step}")
        break

    if info["enemy_altitude"] < info["min_combat_altitude"]:
        winner = "Enemy crashed / below combat altitude"
        print(f"{winner} at step {step}")
        break

else:
    winner = "No winner"
    print(f"{winner} after {MAX_COMBAT_STEPS} steps")


agent_traj = np.array(agent_traj)
enemy_traj = np.array(enemy_traj)
modes = np.array(modes)
actions = np.array(actions)


print("\n========== Combat Summary ==========")
print("winner:", winner)
print("steps:", len(agent_traj))
print("final agent hp:", agent_hp[-1])
print("final enemy hp:", enemy_hp[-1])
print("min agent altitude:", np.min(agent_traj[:, 2]))
print("min enemy altitude:", np.min(enemy_traj[:, 2]))
print("min dist:", np.min(dists))
print("final dist:", dists[-1])
print("min agent LOS:", np.min(agent_los))
print("min enemy LOS:", np.min(enemy_los))
print("max agent damage to HP:", np.max(agent_damage_to_hp))
print("max enemy damage to HP:", np.max(enemy_damage_to_hp))
print("total agent HP damage:", np.sum(agent_damage_to_hp))
print("total enemy HP damage:", np.sum(enemy_damage_to_hp))
print("total reward:", np.sum(rewards))
print("firing event count:", len(firing_events))

print("\n========== Mode Count ==========")
for mode_id, count in Counter(modes).items():
    print(f"{MODE_NAMES[mode_id]:8s}: {count}")

print("\n========== Action Stats ==========")
print("action mean [dir_x, dir_y, dir_z, throttle]:", actions.mean(axis=0))
print("action std  [dir_x, dir_y, dir_z, throttle]:", actions.std(axis=0))


# =========================
# 3D Plot
# =========================
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(agent_traj[:, 0], agent_traj[:, 1], agent_traj[:, 2], label="Recurrent PPO agent")
ax.plot(enemy_traj[:, 0], enemy_traj[:, 1], enemy_traj[:, 2], label="BT enemy")

ax.scatter(agent_traj[0, 0], agent_traj[0, 1], agent_traj[0, 2], s=80, label="agent start")
ax.scatter(enemy_traj[0, 0], enemy_traj[0, 1], enemy_traj[0, 2], s=80, label="enemy start")

ax.scatter(agent_traj[-1, 0], agent_traj[-1, 1], agent_traj[-1, 2], s=80, marker="x", label="agent end")
ax.scatter(enemy_traj[-1, 0], enemy_traj[-1, 1], enemy_traj[-1, 2], s=80, marker="x", label="enemy end")

# firing visualization
for event in firing_events[::max(1, len(firing_events) // 200)]:
    ap = event["agent_pos"]
    ep = event["enemy_pos"]

    ax.scatter(ap[0], ap[1], ap[2], s=20, c="red", alpha=0.6)
    ax.scatter(ep[0], ep[1], ep[2], s=20, c="orange", alpha=0.6)
    ax.plot(
        [ap[0], ep[0]],
        [ap[1], ep[1]],
        [ap[2], ep[2]],
        color="red",
        linewidth=0.8,
        alpha=0.25,
    )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Altitude")
ax.set_title(f"Recurrent PPO Direct Guidance - {winner}")
ax.legend()
plt.show()


# =========================
# Top View
# =========================
plt.figure(figsize=(8, 8))
plt.plot(agent_traj[:, 0], agent_traj[:, 1], label="agent")
plt.plot(enemy_traj[:, 0], enemy_traj[:, 1], label="enemy")

plt.scatter(agent_traj[0, 0], agent_traj[0, 1], s=80, label="agent start")
plt.scatter(enemy_traj[0, 0], enemy_traj[0, 1], s=80, label="enemy start")

plt.scatter(agent_traj[-1, 0], agent_traj[-1, 1], s=80, marker="x", label="agent end")
plt.scatter(enemy_traj[-1, 0], enemy_traj[-1, 1], s=80, marker="x", label="enemy end")

plt.title("Top View")
plt.xlabel("X")
plt.ylabel("Y")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# HP
# =========================
plt.figure(figsize=(8, 4))
plt.plot(agent_hp, label="agent HP")
plt.plot(enemy_hp, label="enemy HP")
plt.title("HP")
plt.xlabel("Step")
plt.ylabel("HP")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# Actual HP Damage
# =========================
plt.figure(figsize=(8, 4))
plt.plot(agent_damage_to_hp, label="agent damage to enemy HP")
plt.plot(enemy_damage_to_hp, label="enemy damage to agent HP")
plt.title("Actual HP Damage per Step")
plt.xlabel("Step")
plt.ylabel("HP Damage")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# LOS
# =========================
plt.figure(figsize=(8, 4))
plt.plot(agent_los, label="agent LOS")
plt.plot(enemy_los, label="enemy LOS")
plt.axhline(10.0, linestyle="--", label="strong cone")
plt.axhline(25.0, linestyle="--", label="mid cone")
plt.axhline(45.0, linestyle="--", label="weak cone")
plt.title("LOS Angle")
plt.xlabel("Step")
plt.ylabel("Degree")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# Distance
# =========================
plt.figure(figsize=(8, 4))
plt.plot(dists)
plt.title("Distance")
plt.xlabel("Step")
plt.ylabel("Distance")
plt.grid(True)
plt.show()


# =========================
# Energy
# =========================
plt.figure(figsize=(8, 4))
plt.plot(agent_energy, label="agent energy")
plt.plot(enemy_energy, label="enemy energy")
plt.title("Energy")
plt.xlabel("Step")
plt.ylabel("Energy")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# Actions
# =========================
plt.figure(figsize=(10, 4))
plt.plot(actions[:, 0], label="dir_x")
plt.plot(actions[:, 1], label="dir_y")
plt.plot(actions[:, 2], label="dir_z")
plt.plot(actions[:, 3], label="throttle")
plt.title("Actions")
plt.xlabel("Step")
plt.ylabel("Action")
plt.grid(True)
plt.legend()
plt.show()


# =========================
# Mode
# =========================
plt.figure(figsize=(10, 3))
plt.plot(modes)
plt.yticks([0, 1, 2, 3], ["RECOVER", "EVADE", "ATTACK", "PURSUIT"])
plt.title("High-level Mode")
plt.xlabel("Step")
plt.ylabel("Mode")
plt.grid(True)
plt.show()